<a href="https://colab.research.google.com/github/BiagioLuc/Visual-Place-Recognition-Project/blob/main/Progetto_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone --recursive https://github.com/BiagioLuc/Visual-Place-Recognition-Project.git

Cloning into 'Visual-Place-Recognition-Project'...
remote: Enumerating objects: 11704, done.
remote: Counting objects: 100% (737/737), done.
remote: Compressing objects: 100% (613/613), done.
fetch-pack: unexpected disconnect while reading sideband packet
^C


In [ ]:
%cd /content/Visual-Place-Recognition-Project/image-matching-models/
!pip install -e .[all]


[Errno 2] No such file or directory: 'content'
/content/Visual-Place-Recognition-Project


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.4 MB/s eta 0:00:00


In [ ]:
%cd /content/Visual-Place-Recognition-Project/
!python download_datasets.py

/content/Visual-Place-Recognition-Project
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 273, in download
    url = get_url_from_gdrive_confirmation(res.text)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gdown/download.py", line 53, in get_url_from_gdrive_confirmation
    raise FileURLRetrievalError(error)
gdown.exceptions.FileURLRetrievalError: Too many users have viewed or downloaded this file recently. Please try accessing the file again later. If the file you are trying to access is particularly large or is shared with many people, it may take up to 24 hours to be able to view or download the file. If you still can't access a file after 24 hours, contact your domain administrator.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/Visual-Place-Recognition-Project/download_datasets.py", line 16, in

In [ ]:
!python VPR-methods-evaluation/main.py \
--num_workers 8 \
--batch_size 32 \
--log_dir log_dir \
--method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
--image_size 512 512 \
--database_folder '<path-to-database-folder>' \
--queries_folder '<path-to-queries-folder>' \
--num_preds_to_save 20 \
--recall_values 1 5 10 20 \
--save_for_uncertainty

In [ ]:
!python match_queries_preds.py \
--preds-dir '/content/Visual-Place-Recognition-Project/logs/log_dir/2026-05-19_15-23-11/preds' \
--matcher 'loftr' \
--device 'cuda' \
--num-preds 20

/content/Visual-Place-Recognition-Project/image-matching-models/matching/third_party/LightGlue/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
100% 44.2M/44.2M [00:02<00:00, 16.2MB/s]
100% 823/823 [51:30<00:00,  3.76s/it]


In [ ]:
!python reranking.py \
--preds-dir '/content/Visual-Place-Recognition-Project/logs/log_dir/2026-05-19_15-23-11/preds' \
--inliers-dir '/content/Visual-Place-Recognition-Project/logs/log_dir/2026-05-19_15-23-11/preds_loftr' \
--num-preds 20 \
--recall-values 1 5 10 20

100% 823/823 [00:01<00:00, 641.02it/s]
R@1: 61.4, R@5: 65.4, R@10: 66.2, R@20: 67.6


## Start extension part

In [ ]:
!python VPR-methods-evaluation/main2.py \
--num_workers 8 \
--batch_size 32 \
--log_dir log_dir \
--method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
--image_size 512 512 \
--database_folder '<path-to-database-folder>' \
--save_descriptors

2026-06-02 20:45:37 VPR-methods-evaluation/main2.py --num_workers 8 --batch_size 32 --log_dir log_dir --method=cosplace --backbone=ResNet18 --descriptors_dimension=512 --image_size 512 512 --database_folder /content/Visual-Place-Recognition-Project/data/gsv_xs/train/washingtondc --save_descriptors
2026-06-02 20:45:37 Arguments: Namespace(positive_dist_threshold=25, method='cosplace', backbone='ResNet18', descriptors_dimension=512, database_folder='/content/Visual-Place-Recognition-Project/data/gsv_xs/train/washingtondc', num_workers=8, batch_size=32, log_dir='log_dir', device='cuda', recall_values=[1, 5, 10, 20], no_labels=False, num_preds_to_save=0, save_only_wrong_preds=False, image_size=[512, 512], save_descriptors=True, save_for_uncertainty=False, use_labels=True)
2026-06-02 20:45:37 Testing with cosplace with a ResNet18 backbone and descriptors dimension 512
2026-06-02 20:45:37 The outputs are being saved in logs/log_dir/2026-06-02_20-45-37
Using cache found in /root/.cache/torch/

In [ ]:
import torch
import os
from pathlib import Path


all_descriptors = torch.load("/content/Visual-Place-Recognition-Project/logs/log_dir/2026-06-02_20-45-37/database_descriptors.torch")

num_esempi = all_descriptors.shape[0]

train_dir = Path("file_train")
val_dir = Path("file_val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

split_idx = int(num_esempi * 0.8) #Splitting the descriptors for traning and validation

train_feats = all_descriptors[:split_idx]
val_feats = all_descriptors[split_idx:]

torch.save(train_feats, train_dir / "train_feats_part22.pt")
torch.save(val_feats, val_dir / "val_feats_part22.pt")

print(f"Train shape: {train_feats.shape} -> Saving in {train_dir}")
print(f"Val shape: {val_feats.shape} -> Saving in {val_dir}")


Train shape: torch.Size([9236, 512]) -> Salvato in file_train
Val shape: torch.Size([2309, 512]) -> Salvato in file_val


In [ ]:
!pip install accelerate circuitsvis datasets diffusers einops huggingface-hub nnsight pandas plotly scikit-learn sentencepiece transformers wandb umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 9.7 MB/s eta 0:00:00


In [ ]:
!git clone --recursive https://github.com/BiagioLuc/sae-for-vlm.git

Cloning into 'sae-for-vlm'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 99 (delta 30), reused 87 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 23.27 MiB | 18.93 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [ ]:
!pip install wandb
!wandb login

In [ ]:
!python sae-for-vlm/sae_train.py \
  --sae_model top_k \
  --activations_dir "/content/file_train" \
  --val_activations_dir "/content/file_val" \
  --device cuda:0 \
  --lr 1e-3 \
  --batch_size 1024 \
  --expansion_factor 2 \
  --steps 11000 \
  --decay_start 5000 \
  --k 128 \
  --seed 222

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
✓ Seed globale impostato a: 222
torch.Size([1024, 4096])
Calculating norm factor:   0% 0/100 [00:00<?, ?it/s]wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
Calculating norm factor: 101it [00:00, 179.91it/s]            
Average mean squared norm: 1.0
Norm factor: 1.0
  0% 0/11000 [00:00<?, ?it/s]wandb: Currently logged in as: trapanottobiagio (trapanottobiagio-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /content/wandb/

In [ ]:
!python sae-for-vlm/save_activations2.py \
  --vpr_descriptors_dir "/content/file_val" \
  --sae_checkpoint "/content/file_train_top_k_128_x2/trainer_0/ae.pt" \
  --output_dir "/content/activationsMixVPR" \
  --k 128 \
  --batch_size 1024

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Utilizzo il dispositivo: cuda:0
Caricamento dei descrittori da: /content/file_val
Estrazione delle attivazioni sparse
→ Matrice delle attivazioni generata! Forma: torch.Size([70012, 8192])
✓ File salvato correttamente in: /content/activationsMixVPR/0.pt



In [ ]:
!python sae-for-vlm/similarity_baseline \
--model "dinov2-base"\
--output_subdir "embeddings"\
--data_path "/content/dataset_split/val"


-> Inizializzazione del modello...
Loading weights: 100% 223/223 [00:00<00:00, 643.36it/s]
-> Avvio del calcolo degli embeddings...
-> Sottocartelle (città) trovate: ['bangkok', 'barcelona', 'boston', 'brussels', 'buenosaires', 'chicago', 'lisbon', 'london', 'losangeles', 'madrid', 'medellin', 'melbourne', 'mexicocity', 'miami', 'minneapolis', 'osaka', 'osl', 'phoenix', 'prg', 'prs', 'rome', 'trt', 'washingtondc']
-> Totale immagini individuate: 105913
-> Spostamento del modello su device: cuda:0
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Processing images i

In [ ]:
!python sae-for-vlm/metric.py \
  --embeddings_path "/content/embeddings/embeddings_dinov2-base.pt" \
  --activations_dir "/content/activations_Cosplace" \
  --output_subdir "risultatiMetrica" \
  --device cuda:0

Utilizzo del device: cuda:0
Loaded embeddings found at /content/embeddings/embeddings_dinov2-base.pt
Embeddings shape: torch.Size([105913, 768])
Loaded activations found at /content/activations_Cosplace
Activations shape: torch.Size([105913, 2048])
Processing image pairs: 100% 105913/105913 [23:01<00:00, 76.68it/s]  
Monosemanticity: 0.25542980432510376 +- 0.0474541150033474
Dead neurons: 0
Total neurons: 2048
Top 10 most monosemantic neurons:
1. Neuron 62 - 0.5025668144226074
2. Neuron 1162 - 0.4765634536743164
3. Neuron 1675 - 0.45252957940101624
4. Neuron 609 - 0.4419957399368286
5. Neuron 923 - 0.4358856678009033
6. Neuron 1622 - 0.428012877702713
7. Neuron 1403 - 0.42526406049728394
8. Neuron 1739 - 0.42396751046180725
9. Neuron 1554 - 0.4140680432319641
10. Neuron 458 - 0.40919506549835205

Bottom 10 least monosemantic neurons:
1. Neuron 969 - 0.13494683802127838
2. Neuron 991 - 0.13827574253082275
3. Neuron 109 - 0.14846274256706238
4. Neuron 1897 - 0.14948932826519012
5. Neuron

In [ ]:
import torch
import numpy as np
import os

# Script to find the 30 neuron indices, relatively at the top 10, medium 10, bottom 10 and their respctively 16 top images (activations)

#Load the scores
path_scores = "/content/activationsMixVPR/risultatiMetricaMixVPR/all_neurons_scores.pth"
monosemanticity = torch.load(path_scores, map_location='cpu')

#Exclude the neurons with values 0
valid_indices = torch.nonzero(~torch.isnan(monosemanticity)).squeeze()
valid_monosemanticity = monosemanticity[valid_indices]

#Sotoing of the vscores
sorted_values, sorted_indices_in_valid = torch.sort(valid_monosemanticity)


# top_10
top_10 = valid_indices[sorted_indices_in_valid[-10:]].flip(dims=[0]).tolist()

# mid_10
mid_point = len(sorted_indices_in_valid) // 2
mid_10 = valid_indices[sorted_indices_in_valid[mid_point - 5 : mid_point + 5]].tolist()

# bottom_10
bottom_10 = valid_indices[sorted_indices_in_valid[:10]].tolist()


selected_neurons = top_10 + mid_10 + bottom_10

print(f"-> Neurons Group 0 (Top 10 più monosemantici): {top_10}")
print(f"-> Neurons Group 1 (Mid 10 intermedi): {mid_10}")
print(f"-> Neurons Group 2 (Bottom 10 meno monosemantici): {bottom_10}")

#Load theactivations
path_activations = "/content/activationsMixVPR/0.pt"
print(f"\nCaricamento delle attivazioni da {path_activations}...")
activations = torch.load(path_activations, map_location='cpu')
print(f"Forma della matrice delle attivazioni: {activations.shape}")

# Extraction the top 16 images for the neurons selected
K = 16
hai_matrix = []

for neuron_id in selected_neurons:
    neuron_activations = activations[:, neuron_id]
    # Taken the values for each neuron, we select the top 16 values
    _, top_k_img_indices = torch.topk(neuron_activations, K, largest=True)
    hai_matrix.append(top_k_img_indices.numpy())

# Convert to matrix (Shape: 30, 16)
hai_matrix = np.array(hai_matrix)

#Saving the matrix
output_dir = "/content"
os.makedirs(output_dir, exist_ok=True)
output_npy_path = os.path.join(output_dir, "hai_matrix.npy")

np.save(output_npy_path, hai_matrix)
print(f"\n✓ File '{output_npy_path}' Successfully generated!")
print(f"Matrix Shape: {hai_matrix.shape} (30 neurons, 16 images)")

-> Neuroni Gruppo 0 (Top 10 più monosemantici): [126, 6486, 6299, 5113, 4500, 57, 1181, 7815, 3346, 4855]
-> Neuroni Gruppo 1 (Mid 10 intermedi): [2051, 389, 1597, 5420, 6994, 1902, 2321, 4175, 4244, 457]
-> Neuroni Gruppo 2 (Bottom 10 meno monosemantici): [3995, 427, 606, 7909, 425, 5014, 6389, 4846, 4759, 2511]

Caricamento delle attivazioni da /content/activationsMixVPR/0.pt...
Forma della matrice delle attivazioni: torch.Size([70012, 8192])

✓ File '/content/hai_matrix.npy' generato con successo!
Forma finale della matrice: (30, 16) (30 neuroni, 16 immagini ciascuno)


In [ ]:
import os
import glob
import zipfile
import math
from pathlib import Path

#Extraction of the images for validation and zipped the file for saving them
base_path = "/content/gsv_xs/train"
output_dir = Path("/content/immagini_val")
os.makedirs(output_dir, exist_ok=True)

cities = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
cities.sort()

print(f"Città processate in ordine alfabetico: {cities}\n")

for city in cities:
    city_path = os.path.join(base_path, city)

    images = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        images.extend(glob.glob(os.path.join(city_path, ext)))

    images.sort()

    totale_immagini_citta = len(images)
    if totale_immagini_citta == 0:
        continue

    num_da_prendere = math.ceil(totale_immagini_citta * 0.20)

    ultimo_20_immagini = images[-num_da_prendere:]

    city_zip_path = output_dir / f"{city}_images_part20.zip"
    print(f"Città: {city:<15} | Immagini totali: {totale_immagini_citta:<5} | Salvate nel file ZIP: {len(ultimo_20_immagini):<5}")

    with zipfile.ZipFile(city_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for img_path in ultimo_20_immagini:
            nome_file_puro = os.path.basename(img_path)
            zipf.write(img_path, arcname=nome_file_puro)

print(f"\nOperation completed. File ZIP saved in: {output_dir}")

Città processate in ordine alfabetico: ['bangkok', 'barcelona', 'boston', 'brussels', 'buenosaires', 'chicago', 'lisbon', 'london', 'losangeles', 'madrid', 'medellin', 'melbourne', 'mexicocity', 'miami', 'minneapolis', 'osaka', 'osl', 'phoenix', 'prg', 'prs', 'rome', 'trt', 'washingtondc']

Città: bangkok         | Immagini totali: 22271 | Salvate nel file ZIP: 4455 
Città: barcelona       | Immagini totali: 15894 | Salvate nel file ZIP: 3179 
Città: boston          | Immagini totali: 32616 | Salvate nel file ZIP: 6524 
Città: brussels        | Immagini totali: 14171 | Salvate nel file ZIP: 2835 
Città: buenosaires     | Immagini totali: 8481  | Salvate nel file ZIP: 1697 
Città: chicago         | Immagini totali: 34091 | Salvate nel file ZIP: 6819 
Città: lisbon          | Immagini totali: 27045 | Salvate nel file ZIP: 5409 
Città: london          | Immagini totali: 58672 | Salvate nel file ZIP: 11735
Città: losangeles      | Immagini totali: 8891  | Salvate nel file ZIP: 1779 
Città:

In [ ]:
import os
import zipfile
import glob


cartella_zip = "/content/immagini_val"

estrazione_path = "/content/dataset_split/val"


file_zip_presenti = glob.glob(os.path.join(cartella_zip, "*.zip"))

for zip_path in file_zip_presenti:
    nome_file = os.path.basename(zip_path)
    nome_citta = nome_file.split("_")[0]
    cartella_citta_dest = os.path.join(estrazione_path, nome_citta)
    os.makedirs(cartella_citta_dest, exist_ok=True)


    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(cartella_citta_dest)
        print(f"Extraction city: {nome_citta} in {cartella_citta_dest}")

print("\nExtraction totally completed")

Estratta città: washingtondc in /content/dataset_split/val/washingtondc
Estratta città: chicago in /content/dataset_split/val/chicago
Estratta città: barcelona in /content/dataset_split/val/barcelona
Estratta città: madrid in /content/dataset_split/val/madrid
Estratta città: minneapolis in /content/dataset_split/val/minneapolis
Estratta città: rome in /content/dataset_split/val/rome
Estratta città: brussels in /content/dataset_split/val/brussels
Estratta città: mexicocity in /content/dataset_split/val/mexicocity
Estratta città: phoenix in /content/dataset_split/val/phoenix
Estratta città: boston in /content/dataset_split/val/boston
Estratta città: prs in /content/dataset_split/val/prs
Estratta città: lisbon in /content/dataset_split/val/lisbon
Estratta città: medellin in /content/dataset_split/val/medellin
Estratta città: losangeles in /content/dataset_split/val/losangeles
Estratta città: prg in /content/dataset_split/val/prg
Estratta città: osl in /content/dataset_split/val/osl
Estrat

In [ ]:
!python sae-for-vlm/visualize_neurons.py \
--output_dir "Visualizzazione" \
--dataset_name "inat" \
--data_path "/content/dataset_split" \
--split "val" \
--group_fractions 0.34 0.34 0.32 \
--hai_indices_path  "/content/hai_matrix.npy"


Loaded HAI indices found at /content/hai_matrix.npy
Visualizing neuron 0 (absolute 0) in group 0
Visualizing neuron 1 (absolute 1) in group 0
Visualizing neuron 2 (absolute 2) in group 0
Visualizing neuron 3 (absolute 3) in group 0
Visualizing neuron 4 (absolute 4) in group 0
Visualizing neuron 5 (absolute 5) in group 0
Visualizing neuron 6 (absolute 6) in group 0
Visualizing neuron 7 (absolute 7) in group 0
Visualizing neuron 8 (absolute 8) in group 0
Visualizing neuron 9 (absolute 9) in group 0
Visualizing neuron 0 (absolute 10) in group 1
Visualizing neuron 1 (absolute 11) in group 1
Visualizing neuron 2 (absolute 12) in group 1
Visualizing neuron 3 (absolute 13) in group 1
Visualizing neuron 4 (absolute 14) in group 1
Visualizing neuron 5 (absolute 15) in group 1
Visualizing neuron 6 (absolute 16) in group 1
Visualizing neuron 7 (absolute 17) in group 1
Visualizing neuron 8 (absolute 18) in group 1
Visualizing neuron 9 (absolute 19) in group 1
Visualizing neuron 0 (absolute 20) in 